---
## Phase 1 — Environment Setup & Data Ingestion

In [ ]:
# ── Phase 1: Imports & Configuration ──────────────────────────────────────────
import polars as pl
import numpy as np
import joblib
import os
import psycopg2
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN

# ── Database connection ───────────────────────────────────────────────────────
# Using psycopg2 connection dicts (bypasses ConnectorX entirely)
DATABASE_CONN = dict(
    host="host_url",
    port=5432,
    dbname="capstone_db",
    user="root",
    password="password",
)
TRADE_CONN = dict(
    host="host_url",
    port=5432,
    dbname="mydb",
    user="root",
    password="password",
)

# ── Pipeline parameters ──────────────────────────────────────────────────────
TICKERS             = ["SPY", "QQQ", "NVDA"]  # Each ticker gets its own model
ROLLING_WINDOW_DAYS = 21              # Train on the last N days of data
DEFAULT_CONTAMINATION = 0.005         # Fallback anomaly fraction for unmapped tickers
CONTAMINATION_BY_TICKER = {
    "SPY": 0.0035,   # Lower than before because SPY flow is very dense
    "QQQ": 0.0045,
    "NVDA": 0.0060,
}
RISK_FREE_RATE      = 0.05            # Annualised risk-free rate for Black-Scholes
ROLLING_MINUTES     = 15              # Window for rolling aggressiveness metrics
OUTPUT_DIR          = "models"        # Directory for exported model files

# ── Event grouping parameters (offline) ─────────────────────────────────────
DBSCAN_EPS_SECONDS = 180              # Time radius for anomaly burst grouping
DBSCAN_EPS_SCORE   = 0.20             # Score radius (decision_function space)
DBSCAN_MIN_SAMPLES = 3                # Min points to form a cluster
MIN_EVENT_TRADES   = 3                # Filter tiny clusters after grouping

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Configuration loaded  |  Tickers: {TICKERS}")
print(f"✓ Contamination map: {CONTAMINATION_BY_TICKER}")

✓ Configuration loaded  |  Tickers: ['SPY', 'QQQ', 'NVDA']
✓ Contamination map: {'SPY': 0.0035, 'QQQ': 0.0045, 'NVDA': 0.006}


In [2]:
# ── Phase 1: Data Ingestion ───────────────────────────────────────────────────
# Load trades joined with symbol metadata (expiration, call/put, strike, root)
# Uses psycopg2 connections (more reliable than ConnectorX for remote hosts).

def load_trades(conn_params: dict, start_date: str, end_date: str, root: str) -> pl.DataFrame:
    """
    Load trades joined with symbol metadata for a given ticker and date range.
    Returns a Polars DataFrame ready for feature engineering.
    """
    query = """
        SELECT
            t.trade_id,
            t.symbol,
            t.time,
            t.bid,
            t.ask,
            t.trade_price,
            t.trade_size,
            t.trade_condition,
            s.root,
            s.expiration,
            s.call_putt,
            s.strike
        FROM trades t
        JOIN symbols s ON t.symbol = s.symbol
        WHERE t.time >= %s
          AND t.time <  %s
          AND s.root = %s
        ORDER BY t.time
    """
    conn = psycopg2.connect(**conn_params)
    df = pl.read_database(query, conn, execute_options={"parameters": [start_date, end_date, root]})
    print(f"  Loaded {df.height:,} trades  |  {df.estimated_size('mb'):.1f} MB")
    print(df)
    return df


def load_market_data(conn_params: dict, start_date: str, end_date: str, root: str) -> pl.DataFrame:
    """
    Load underlying price data (1-min bars) from the market_data table
    for a specific ticker.
    """
    query = """
        SELECT time, symbol, close
        FROM market_data
        WHERE time >= %s
          AND time <  %s
          AND symbol = %s
        ORDER BY time
    """
    conn = psycopg2.connect(**conn_params)
    try:
        df = pl.read_database(query, conn, execute_options={"parameters": [start_date, end_date, root]})
    finally:
        conn.close()
    print(f"  Loaded {df.height:,} market bars  |  {df.estimated_size('mb'):.1f} MB")
    return df

print("✓ Ingestion functions defined")

✓ Ingestion functions defined


In [3]:
# ── Phase 1: Execute Ingestion (date window) ─────────────────────────────────
end_date   = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=ROLLING_WINDOW_DAYS+1)).strftime("%Y-%m-%d")
start_date = '2026-02-09'
end_date = '2026-03-02'
print(f"Date window: {start_date} → {end_date}")
print(f"Tickers to process: {TICKERS}")

Date window: 2026-02-09 → 2026-03-02
Tickers to process: ['SPY', 'QQQ', 'NVDA']


---
## Phase 2 — Data Transformation & Feature Engineering

In [4]:
# ── Phase 2a: Basic Microstructure, DTE, Categorical Encoding ─────────────────

def basic_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    Add spread, midpoint, time-to-expiry, and categorical encodings.
    """
    df = df.with_columns([
        # ── Microstructure ──
        (pl.col("ask") - pl.col("bid")).alias("spread"),
        ((pl.col("ask") + pl.col("bid")) / 2.0).alias("midpoint"),

        # ── Time-to-expiry (fractional days & annualised years) ──
        # Ensure both columns are proper datetimes for subtraction
        pl.col("time").cast(pl.Datetime("us")),
        pl.col("expiration").cast(pl.Datetime("us")),
    ]).with_columns([
        (
            (pl.col("expiration") - pl.col("time"))
            .dt.total_seconds() / 86_400.0
        ).alias("dte_days"),
    ]).with_columns([
        (pl.col("dte_days") / 365.25).alias("dte_years"),
    ]).with_columns([
        # ── Categorical: Call/Put → binary ──
        pl.when(pl.col("call_putt") == "call")
          .then(1)
          .otherwise(0)
          .alias("is_call"),

        # ── One-Hot Encode trade_condition ──
        (pl.col("trade_condition") == "buy").cast(pl.Int8).alias("tc_buy"),
        (pl.col("trade_condition") == "sell").cast(pl.Int8).alias("tc_sell"),
        (pl.col("trade_condition") == "strong_buy").cast(pl.Int8).alias("tc_strong_buy"),
        (pl.col("trade_condition") == "strong_sell").cast(pl.Int8).alias("tc_strong_sell"),
    ])
    return df

print("✓ basic_features() defined")

✓ basic_features() defined


In [5]:
# ── Phase 2b: Merge Underlying Price & Compute Moneyness ──────────────────────
# The market_data table has 1-min close prices keyed by (time, symbol).
# We do an asof-join: for each option trade, find the most recent underlying
# bar whose timestamp is ≤ the trade's timestamp.

def merge_underlying_price(
    trades: pl.DataFrame,
    market: pl.DataFrame,
) -> pl.DataFrame:
    """
    Asof-join market bars onto trades by (root ↔ symbol) and time.
    Adds 'underlying_price' and 'moneyness' columns.
    """
    # Rename market columns for the join
    market = (
        market
        .rename({"close": "underlying_price", "symbol": "root"})
        .sort("root", "time")
        .with_columns(pl.col("time").cast(pl.Datetime("us")))
    )

    trades = trades.sort("root", "time")

    # Asof join: each trade gets the latest available underlying price
    joined = trades.join_asof(
        market.select(["root", "time", "underlying_price"]),
        on="time",
        by="root",
        strategy="backward",
    )

    # Moneyness = strike / underlying_price
    joined = joined.with_columns([
        (pl.col("strike") / pl.col("underlying_price")).alias("moneyness"),
    ])

    return joined

print("✓ merge_underlying_price() defined")

✓ merge_underlying_price() defined


In [6]:
# ── Phase 2c: Black-Scholes Greeks ────────────────────────────────────────────
# Vectorised computation of IV, Delta, Gamma, Theta using scipy.

from scipy.stats import norm

def _d1d2(S, K, T, r, sigma):
    """Compute d1 and d2 for Black-Scholes."""
    T = np.maximum(T, 1e-10)  # avoid division by zero
    sigma = np.maximum(sigma, 1e-10)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2


def bs_price(S, K, T, r, sigma, is_call):
    """Vectorised Black-Scholes price."""
    d1, d2 = _d1d2(S, K, T, r, sigma)
    call_price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    put_price  = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    return np.where(is_call, call_price, put_price)


def bs_iv(market_price, S, K, T, r, is_call, tol=1e-6, max_iter=100):
    """Newton-Raphson implied volatility solver (vectorised)."""
    sigma = np.full_like(market_price, 0.3)  # initial guess
    for _ in range(max_iter):
        price = bs_price(S, K, T, r, sigma, is_call)
        d1, _ = _d1d2(S, K, T, r, sigma)
        vega = S * norm.pdf(d1) * np.sqrt(np.maximum(T, 1e-10))
        vega = np.maximum(vega, 1e-10)
        diff = price - market_price
        sigma = sigma - diff / vega
        sigma = np.clip(sigma, 1e-6, 10.0)
        if np.all(np.abs(diff) < tol):
            break
    return sigma


def bs_delta(S, K, T, r, sigma, is_call):
    d1, _ = _d1d2(S, K, T, r, sigma)
    return np.where(is_call, norm.cdf(d1), norm.cdf(d1) - 1)


def bs_gamma(S, K, T, r, sigma):
    d1, _ = _d1d2(S, K, T, r, sigma)
    return norm.pdf(d1) / (S * sigma * np.sqrt(np.maximum(T, 1e-10)))


def bs_theta(S, K, T, r, sigma, is_call):
    d1, d2 = _d1d2(S, K, T, r, sigma)
    term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(np.maximum(T, 1e-10)))
    call_theta = term1 - r * K * np.exp(-r * T) * norm.cdf(d2)
    put_theta  = term1 + r * K * np.exp(-r * T) * norm.cdf(-d2)
    return np.where(is_call, call_theta, put_theta) / 365.0  # per-day


def compute_greeks(df: pl.DataFrame, r: float = RISK_FREE_RATE) -> pl.DataFrame:
    """
    Compute IV, Delta, Gamma, Theta and attach as new columns.
    Requires: underlying_price, strike, dte_years, midpoint, is_call.
    """
    # Drop rows where underlying price could not be resolved
    df = df.filter(pl.col("underlying_price").is_not_null())

    S   = df["underlying_price"].to_numpy().astype(np.float64)
    K   = df["strike"].to_numpy().astype(np.float64)
    T   = df["dte_years"].to_numpy().astype(np.float64)
    mid = df["midpoint"].to_numpy().astype(np.float64)
    ic  = df["is_call"].to_numpy().astype(bool)

    # Clamp T to avoid degenerate values
    T = np.maximum(T, 1e-6)

    # Use midpoint as the "market price" for IV extraction
    iv    = bs_iv(mid, S, K, T, r, ic)
    delta = bs_delta(S, K, T, r, iv, ic)
    gamma = bs_gamma(S, K, T, r, iv)
    theta = bs_theta(S, K, T, r, iv, ic)

    df = df.with_columns([
        pl.Series("iv",    iv,    dtype=pl.Float64),
        pl.Series("delta", delta, dtype=pl.Float64),
        pl.Series("gamma", gamma, dtype=pl.Float64),
        pl.Series("theta", theta, dtype=pl.Float64),
    ])
    return df

print("✓ Greeks functions defined")

✓ Greeks functions defined


In [7]:
# ── Phase 2d: Rolling Aggressiveness Metrics ──────────────────────────────────
# For each (symbol, strike) group, compute a trailing window ratio:
#   strong_buy_volume / total_volume

def rolling_aggressiveness(
    df: pl.DataFrame,
    window_minutes: int = ROLLING_MINUTES,
) -> pl.DataFrame:
    """
    Compute the ratio of strong-buy volume to total volume over a trailing
    window, grouped by (symbol, strike).
    """
    window = f"{window_minutes}m"

    # Pre-compute volume contributions
    df = df.with_columns([
        (pl.col("tc_strong_buy") * pl.col("trade_size"))
            .alias("strong_buy_vol"),
    ]).sort("symbol", "time")

    # Rolling sums within each (symbol) group
    rolled = df.rolling(
        index_column="time",
        period=window,
        group_by=["symbol"],
    ).agg([
        pl.col("strong_buy_vol").sum().alias("roll_strong_buy_vol"),
        pl.col("trade_size").sum().alias("roll_total_vol"),
    ])

    # Join rolling aggregates back
    df = df.join(
        rolled,
        on=["symbol", "time"],
        how="left",
    )

    # Ratio (guard against div-by-zero)
    df = df.with_columns([
        (
            pl.col("roll_strong_buy_vol")
            / pl.col("roll_total_vol").clip(lower_bound=1)
        ).alias("aggression_ratio"),
    ])

    return df

print("✓ rolling_aggressiveness() defined")

✓ rolling_aggressiveness() defined


---
## Phase 3 — Model Configuration & Training (IsolationForest)

In [8]:
# ── Phase 3: Feature Selection & Model Definition ─────────────────────────────

# The numerical feature columns fed into IsolationForest
FEATURE_COLS = [
    "spread",
    "midpoint",
    "trade_price",
    "trade_size",
    "dte_days",
    "dte_years",
    "is_call",
    "tc_buy",
    "tc_sell",
    "tc_strong_buy",
    "tc_strong_sell",
    "moneyness",
    "iv",
    "delta",
    "gamma",
    "theta",
    "aggression_ratio",
]

def prepare_features(df: pl.DataFrame, feature_cols: list[str]) -> np.ndarray:
    """
    Select feature columns, drop any remaining nulls, and convert to a
    NumPy float32 array suitable for scikit-learn.
    """
    features_df = df.select(feature_cols).drop_nulls()
    print(f"  Feature matrix: {features_df.shape[0]:,} rows × {features_df.shape[1]} cols")
    print(f"  Memory: {features_df.estimated_size('mb'):.1f} MB")
    return features_df.to_numpy().astype(np.float32)


def build_model(contamination: float = DEFAULT_CONTAMINATION) -> IsolationForest:
    """
    Instantiate and return a memory-efficient IsolationForest.

    Key parameters for constrained environments:
      • max_samples=256  → limits subsample per tree (fast, low RAM)
      • n_estimators=100 → 100 trees (good accuracy)
      • n_jobs=-1         → use all CPU cores
    """
    model = IsolationForest(
        n_estimators=100,
        max_samples=256,
        contamination=contamination,
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )
    return model

print("✓ Feature selection & model builder defined")

✓ Feature selection & model builder defined


---
## Phase 4 — Output, Scoring & Incremental Daily Updates

In [9]:
# ── Phase 4a: Scoring Function ─────────────────────────────────────────────────

def score_trades(
    df: pl.DataFrame,
    model: IsolationForest,
    feature_cols: list[str],
) -> pl.DataFrame:
    """
    Score every trade with the fitted IsolationForest.
    Adds:
      • anomaly_label  :  -1 (anomaly) or 1 (normal)
      • anomaly_score  :  continuous score from decision_function()
                          (more negative → more anomalous)
    """
    # Identify rows that survived null-drop
    valid_idx = (
        df.select(feature_cols)
          .with_row_index("_idx")
          .drop_nulls()["_idx"]
    )
    X = df.select(feature_cols).drop_nulls().to_numpy().astype(np.float32)

    labels = model.predict(X)
    scores = model.decision_function(X)

    # Build result columns (null for rows that had missing features)
    label_series = pl.Series("anomaly_label", [None] * df.height, dtype=pl.Int8)
    score_series = pl.Series("anomaly_score", [None] * df.height, dtype=pl.Float64)

    # Scatter valid values into correct positions
    idx_np = valid_idx.to_numpy()
    label_arr = label_series.to_numpy(allow_copy=True).astype(np.float64)
    score_arr = score_series.to_numpy(allow_copy=True).astype(np.float64)
    label_arr[idx_np] = labels
    score_arr[idx_np] = scores

    df = df.with_columns([
        pl.Series("anomaly_label", label_arr, dtype=pl.Int8),
        pl.Series("anomaly_score", score_arr, dtype=pl.Float64),
    ])
    return df

print("✓ score_trades() defined")

✓ score_trades() defined


In [10]:
# ── Phase 4b: Per-Ticker Training Pipeline ────────────────────────────────────
# Iterates over each ticker, trains an independent IsolationForest model,
# scores the trades, exports the model and anomalies.

def train_ticker(
    ticker: str,
    db_conn: dict,
    trade_conn: dict,
    start_date: str,
    end_date: str,
    contamination: float | None = None,
    risk_free_rate: float = RISK_FREE_RATE,
) -> tuple[pl.DataFrame, IsolationForest]:
    """
    Full pipeline for a single ticker:
      1. Load trades + market data filtered to this ticker
      2. Engineer features
      3. Train IsolationForest
      4. Score trades
      Returns (scored_df, fitted_model)
    """
    effective_contamination = (
        contamination
        if contamination is not None
        else CONTAMINATION_BY_TICKER.get(ticker, DEFAULT_CONTAMINATION)
    )

    print(f"\n[1/6] Loading {ticker} trades …")
    df = load_trades(trade_conn, start_date, end_date, root=ticker)

    if df.height == 0:
        print(f"  ⚠ No trades found for {ticker} — skipping")
        return None, None

    print(f"[2/6] Loading {ticker} market data …")
    mkt = load_market_data(db_conn, start_date, end_date, root=ticker)

    print(f"[3/6] Engineering features …")
    df = basic_features(df)
    df = merge_underlying_price(df, mkt)
    df = compute_greeks(df, r=risk_free_rate)
    df = rolling_aggressiveness(df)

    print(f"[4/6] Preparing feature matrix …")
    X = prepare_features(df, FEATURE_COLS)

    print(f"[5/6] Training IsolationForest (contamination={effective_contamination:.4f}) …")
    mdl = build_model(contamination=effective_contamination)
    mdl.fit(X)

    print(f"[6/6] Scoring …")
    df = score_trades(df, mdl, FEATURE_COLS)

    n_anom = df.filter(pl.col("anomaly_label") == -1).height
    n_tot  = df.filter(pl.col("anomaly_label").is_not_null()).height
    print(
        f"✓ {ticker} complete  |  contamination={effective_contamination:.4f}"
        f"  |  {n_tot:,} scored  |  {n_anom:,} anomalies ({n_anom / max(n_tot, 1) * 100:.2f}%)"
    )

    return df, mdl

print("✓ train_ticker() defined")

✓ train_ticker() defined


In [11]:
# ── Phase 4c: Train All Tickers, Export Anomalies, Export Grouped Events ─────
# Model files:    models/{TICKER}_{YYYY-MM-DD}.joblib
# Anomaly files:  models/{TICKER}_{YYYY-MM-DD}_anomalies.parquet
# Event files:    models/{TICKER}_{YYYY-MM-DD}_events.parquet

def group_anomalies_to_events(
    anomalies_df: pl.DataFrame,
    eps_seconds: float = DBSCAN_EPS_SECONDS,
    eps_score: float = DBSCAN_EPS_SCORE,
    min_samples: int = DBSCAN_MIN_SAMPLES,
    min_event_trades: int = MIN_EVENT_TRADES,
) -> pl.DataFrame:
    """
    Collapse anomaly trades into burst-like events.

    Grouping strategy:
      1) Split by contract context (root/symbol/expiry/call-put/strike)
      2) DBSCAN over 2D points: [time_seconds / eps_seconds, score / eps_score]
      3) Aggregate each cluster into one event row
    """
    event_schema = {
        "event_id": pl.UInt32,
        "root": pl.Utf8,
        "symbol": pl.Utf8,
        "expiration": pl.Date,
        "call_putt": pl.Utf8,
        "strike": pl.Float64,
        "event_start": pl.Datetime,
        "event_end": pl.Datetime,
        "duration_seconds": pl.Int64,
        "n_trades": pl.UInt32,
        "total_size": pl.Float64,
        "mean_anomaly_score": pl.Float64,
        "worst_anomaly_score": pl.Float64,
        "dominant_trade_condition": pl.Utf8,
    }

    if anomalies_df.height == 0:
        return pl.DataFrame(schema=event_schema)

    working = anomalies_df.with_columns(
        pl.col("time").dt.epoch(time_unit="s").cast(pl.Float64).alias("_time_s")
    ).sort(["root", "symbol", "expiration", "call_putt", "strike", "time"])

    grouped_events: list[pl.DataFrame] = []

    for contract_df in working.partition_by(
        ["root", "symbol", "expiration", "call_putt", "strike"],
        maintain_order=True,
    ):
        cluster_input = contract_df.drop_nulls(["_time_s", "anomaly_score"]) 
        if cluster_input.height < min_samples:
            continue

        X = np.column_stack([
            cluster_input["_time_s"].to_numpy() / max(eps_seconds, 1e-9),
            cluster_input["anomaly_score"].to_numpy() / max(eps_score, 1e-9),
        ])
        labels = DBSCAN(eps=1.0, min_samples=min_samples).fit_predict(X)

        clustered = cluster_input.with_columns(
            pl.Series("_cluster_id", labels, dtype=pl.Int32)
        ).filter(pl.col("_cluster_id") != -1)

        if clustered.height == 0:
            continue

        events = (
            clustered
            .group_by(["root", "symbol", "expiration", "call_putt", "strike", "_cluster_id"])
            .agg([
                pl.col("time").min().alias("event_start"),
                pl.col("time").max().alias("event_end"),
                pl.len().cast(pl.UInt32).alias("n_trades"),
                pl.col("trade_size").sum().cast(pl.Float64).alias("total_size"),
                pl.col("anomaly_score").mean().alias("mean_anomaly_score"),
                pl.col("anomaly_score").min().alias("worst_anomaly_score"),
                pl.col("trade_condition").drop_nulls().mode().first().alias("dominant_trade_condition"),
            ])
            .with_columns(
                (pl.col("event_end").dt.epoch(time_unit="s") - pl.col("event_start").dt.epoch(time_unit="s"))
                .cast(pl.Int64)
                .alias("duration_seconds")
            )
            .filter(pl.col("n_trades") >= min_event_trades)
            .drop("_cluster_id")
        )

        if events.height > 0:
            grouped_events.append(events)

    if not grouped_events:
        return pl.DataFrame(schema=event_schema)

    out = (
        pl.concat(grouped_events, how="vertical_relaxed")
        .sort(["root", "event_start", "event_end"])
        .with_row_index("event_id", offset=1)
        .with_columns(pl.col("event_id").cast(pl.UInt32))
        .select(list(event_schema.keys()))
    )
    return out


today_str = datetime.now().strftime("%Y-%m-%d")

# Store results for optional further analysis
results: dict[str, tuple[pl.DataFrame, IsolationForest]] = {}

for ticker in TICKERS:
    print(f"\n{'═' * 60}")
    print(f"  Training model for: {ticker}")
    print(f"{'═' * 60}")

    scored_df, fitted_model = train_ticker(
        ticker=ticker,
        db_conn=DATABASE_CONN,
        trade_conn=TRADE_CONN,
        start_date=start_date,
        end_date=end_date,
    )

    if scored_df is None:
        continue

    results[ticker] = (scored_df, fitted_model)

    # ── Export model ──────────────────────────────────────────────────────
    model_path = os.path.join(OUTPUT_DIR, f"{ticker}_{today_str}.joblib")
    joblib.dump(fitted_model, model_path)
    print(f"  💾 Model saved → {model_path}")

    # ── Export anomalies ──────────────────────────────────────────────────
    anomalies_df = (
        scored_df
        .filter(pl.col("anomaly_label") == -1)
        .sort("anomaly_score")
    )
    anomalies_path = os.path.join(OUTPUT_DIR, f"{ticker}_{today_str}_anomalies.parquet")
    anomalies_df.write_parquet(anomalies_path)
    print(f"  💾 Anomalies saved → {anomalies_path}  ({anomalies_df.height:,} rows)")

    # ── Group anomalies into events and export ───────────────────────────
    events_df = group_anomalies_to_events(anomalies_df)
    events_path = os.path.join(OUTPUT_DIR, f"{ticker}_{today_str}_events.parquet")
    events_df.write_parquet(events_path)
    print(f"  💾 Events saved → {events_path}  ({events_df.height:,} rows)")

    # ── Quick summary ─────────────────────────────────────────────────────
    print(f"\n  ── {ticker} Anomaly Breakdown by Trade Condition ──")
    summary = (
        anomalies_df
        .group_by("trade_condition")
        .agg([
            pl.len().alias("count"),
            pl.col("trade_size").sum().alias("total_volume"),
            pl.col("anomaly_score").mean().alias("avg_score"),
        ])
        .sort("count", descending=True)
    )
    print(summary)

print(f"\n{'═' * 60}")
print(f"✓ All tickers processed  |  Models saved to '{OUTPUT_DIR}/'")
print(f"  Tickers trained: {list(results.keys())}")
print(f"{'═' * 60}")


════════════════════════════════════════════════════════════
  Training model for: SPY
════════════════════════════════════════════════════════════

[1/6] Loading SPY trades …


  Loaded 13,821,461 trades  |  1216.5 MB
shape: (13_821_461, 12)
┌───────────┬───────────────┬───────────────┬──────┬───┬──────┬───────────────┬───────────┬────────┐
│ trade_id  ┆ symbol        ┆ time          ┆ bid  ┆ … ┆ root ┆ expiration    ┆ call_putt ┆ strike │
│ ---       ┆ ---           ┆ ---           ┆ ---  ┆   ┆ ---  ┆ ---           ┆ ---       ┆ ---    │
│ i64       ┆ str           ┆ datetime[μs]  ┆ f64  ┆   ┆ str  ┆ datetime[μs]  ┆ str       ┆ f64    │
╞═══════════╪═══════════════╪═══════════════╪══════╪═══╪══════╪═══════════════╪═══════════╪════════╡
│ 243823998 ┆ SPY260209C006 ┆ 2026-02-09    ┆ 1.17 ┆ … ┆ SPY  ┆ 2026-02-09    ┆ call      ┆ 691.0  │
│           ┆ 91000         ┆ 08:30:04.276  ┆      ┆   ┆      ┆ 00:00:00      ┆           ┆        │
│ 243824042 ┆ SPY260209P006 ┆ 2026-02-09    ┆ 4.71 ┆ … ┆ SPY  ┆ 2026-02-09    ┆ put       ┆ 694.0  │
│           ┆ 94000         ┆ 08:30:04.553  ┆      ┆   ┆      ┆ 00:00:00      ┆           ┆        │
│ 243823995 ┆ SPY260209P00

/tmp/ipykernel_8917/1240807151.py:25: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  joined = trades.join_asof(


[4/6] Preparing feature matrix …
  Feature matrix: 22,019,041 rows × 17 cols
  Memory: 2183.9 MB
[5/6] Training IsolationForest (contamination=0.0035) …
[6/6] Scoring …
✓ SPY complete  |  contamination=0.0035  |  22,019,041 scored  |  77,065 anomalies (0.35%)
  💾 Model saved → models/SPY_2026-04-13.joblib
  💾 Anomalies saved → models/SPY_2026-04-13_anomalies.parquet  (77,065 rows)
  💾 Events saved → models/SPY_2026-04-13_events.parquet  (4,506 rows)

  ── SPY Anomaly Breakdown by Trade Condition ──
shape: (5, 4)
┌─────────────────┬───────┬──────────────┬───────────┐
│ trade_condition ┆ count ┆ total_volume ┆ avg_score │
│ ---             ┆ ---   ┆ ---          ┆ ---       │
│ str             ┆ u32   ┆ i64          ┆ f64       │
╞═════════════════╪═══════╪══════════════╪═══════════╡
│ buy             ┆ 32218 ┆ 591051       ┆ -0.020346 │
│ sell            ┆ 27332 ┆ 445050       ┆ -0.022422 │
│ strong_buy      ┆ 8996  ┆ 293208       ┆ -0.022801 │
│ strong_sell     ┆ 8099  ┆ 145803       ┆

/tmp/ipykernel_8917/1240807151.py:25: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  joined = trades.join_asof(


[4/6] Preparing feature matrix …
  Feature matrix: 13,213,223 rows × 17 cols
  Memory: 1310.5 MB
[5/6] Training IsolationForest (contamination=0.0045) …
[6/6] Scoring …
✓ QQQ complete  |  contamination=0.0045  |  13,213,223 scored  |  59,460 anomalies (0.45%)
  💾 Model saved → models/QQQ_2026-04-13.joblib
  💾 Anomalies saved → models/QQQ_2026-04-13_anomalies.parquet  (59,460 rows)
  💾 Events saved → models/QQQ_2026-04-13_events.parquet  (2,933 rows)

  ── QQQ Anomaly Breakdown by Trade Condition ──
shape: (5, 4)
┌─────────────────┬───────┬──────────────┬───────────┐
│ trade_condition ┆ count ┆ total_volume ┆ avg_score │
│ ---             ┆ ---   ┆ ---          ┆ ---       │
│ str             ┆ u32   ┆ i64          ┆ f64       │
╞═════════════════╪═══════╪══════════════╪═══════════╡
│ sell            ┆ 25007 ┆ 436042       ┆ -0.014618 │
│ buy             ┆ 18287 ┆ 393773       ┆ -0.014137 │
│ strong_buy      ┆ 11417 ┆ 354078       ┆ -0.029965 │
│ strong_sell     ┆ 4414  ┆ 79844        ┆

/tmp/ipykernel_8917/1240807151.py:25: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  joined = trades.join_asof(


[4/6] Preparing feature matrix …
  Feature matrix: 4,801,095 rows × 17 cols
  Memory: 476.2 MB
[5/6] Training IsolationForest (contamination=0.0060) …
[6/6] Scoring …
✓ NVDA complete  |  contamination=0.0060  |  4,801,095 scored  |  28,803 anomalies (0.60%)
  💾 Model saved → models/NVDA_2026-04-13.joblib
  💾 Anomalies saved → models/NVDA_2026-04-13_anomalies.parquet  (28,803 rows)
  💾 Events saved → models/NVDA_2026-04-13_events.parquet  (792 rows)

  ── NVDA Anomaly Breakdown by Trade Condition ──
shape: (5, 4)
┌─────────────────┬───────┬──────────────┬───────────┐
│ trade_condition ┆ count ┆ total_volume ┆ avg_score │
│ ---             ┆ ---   ┆ ---          ┆ ---       │
│ str             ┆ u32   ┆ i64          ┆ f64       │
╞═════════════════╪═══════╪══════════════╪═══════════╡
│ buy             ┆ 12282 ┆ 680835       ┆ -0.016339 │
│ sell            ┆ 10294 ┆ 512062       ┆ -0.016589 │
│ strong_buy      ┆ 4026  ┆ 166469       ┆ -0.018129 │
│ strong_sell     ┆ 2163  ┆ 176116       ┆